# SGFND - Synthetic Generative Framework Neural Dynamics (Colab)

This notebook compiles and runs the SGFND C project on **Google Colab**.

Features:
- Bridge-based feature representation (92→46 compressed bridges)
- Latent diffusion model (512-dim, 1000 timesteps)
- Tiled rendering (64×64 tiles)
- Large model streaming (30,000 bridges)
- NSFW content filtering (prompt + image analysis)

**Colab provides free x86_64 CPU/GPU - perfect for this project.**

In [ ]:
# ===== Colab Setup =====
# Mount Google Drive if you want persistent storage (optional)
# from google.colab import drive
# drive.mount('/content/drive')

# Install system dependencies (Colab has gcc, make)
!apt-get update -qq && apt-get install -y -qq libcurl4-openssl-dev libcjson-dev 2>/dev/null | tail -3

# Clone from GitHub
GIT_REPO = 'https://github.com/wippsanrinthailand80-commits/sgfnd.git'
!git clone {GIT_REPO} sgfnd
%cd sgfnd

# Verify
!ls -la

In [ ]:
# ===== Build (optimized for x86_64) =====
%%bash
set -e
make clean
make
echo 'Build complete. Binary:'
ls -lh sgfnd

# Show all available options
./sgfnd --help

In [ ]:
# ===== Standard Generation (Small Model) =====
%%bash
./sgfnd --generate
echo 'Output files:'
ls -lh *.raw *.png 2>/dev/null

In [ ]:
# ===== Display Generated Image =====
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def load_raw_image(path):
    with open(path, 'rb') as f:
        header = np.fromfile(f, dtype=np.uint32, count=4)
        w, h, c, bit_depth = header
        data = np.fromfile(f, dtype=np.uint8)
        return data.reshape(h, w, c)

img = load_raw_image('generated_cuda_image.raw')
print(f'Shape: {img.shape}, dtype: {img.dtype}, range: [{img.min()}, {img.max()}]')

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title('SGFND Generated Image (Small Model)')
plt.show()

In [ ]:
# ===== Tiled Streaming Renderer =====
%%bash
./sgfnd --generate --tiled
ls -lh generated_tiled.raw

In [ ]:
# ===== Display Tiled Output =====
img_tiled = load_raw_image('generated_tiled.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img_tiled)
plt.axis('off')
plt.title('SGFND Tiled Streaming Render')
plt.show()

In [ ]:
# ===== Large Model (30,000 bridges) =====
%%bash
./sgfnd --generate --large
ls -lh generated_cuda_image.raw

In [ ]:
# ===== Display Large Model Output =====
img_large = load_raw_image('generated_cuda_image.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img_large)
plt.axis('off')
plt.title('SGFND Large Model (30K Bridges)')
plt.show()

## NSFW Filter Testing

The NSFW filter has three modes:
- `disabled` (default) - no filtering
- `enabled` - checks prompts and images, logs flags
- `strict` - same as enabled but with lower default threshold

Options:
- `--nsfw MODE` - disabled|enabled|strict
- `--nsfw-thresh F` - threshold 0.0-1.0 (default: 0.5)

In [ ]:
# ===== NSFW Filter: Small Model (Enabled) =====
%%bash
./sgfnd --generate --small --nsfw enabled
echo '---'
ls -lh generated_cuda_image.raw

In [ ]:
# ===== Display NSFW Small =====
img = load_raw_image('generated_cuda_image.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Small Model + NSFW Enabled')
plt.show()

In [ ]:
# ===== NSFW Filter: Large Model (Strict) =====
%%bash
./sgfnd --generate --large --nsfw strict --nsfw-thresh 0.3
echo '---'
ls -lh generated_cuda_image.raw

In [ ]:
# ===== Display NSFW Large Strict =====
img = load_raw_image('generated_cuda_image.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Large Model + NSFW Strict (thresh=0.3)')
plt.show()

In [ ]:
# ===== NSFW Filter: Tiled Renderer =====
%%bash
./sgfnd --generate --tiled --nsfw enabled
echo '---'
ls -lh generated_tiled.raw

In [ ]:
# ===== Display NSFW Tiled =====
img = load_raw_image('generated_tiled.raw')
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Tiled Renderer + NSFW Enabled')
plt.show()

In [ ]:
# ===== Combined: Large + Tiled + NSFW Strict =====
%%bash
./sgfnd --generate --large --tiled --nsfw strict --nsfw-thresh 0.5
echo '---'
ls -lh generated_tiled.raw generated_cuda_image.raw 2>/dev/null

In [ ]:
# ===== Memory Check with Valgrind =====
%%bash
apt-get install -y -qq valgrind 2>/dev/null
valgrind --tool=memcheck --error-exitcode=1 ./sgfnd --generate --nsfw enabled 2>&1 | tail -20

In [ ]:
# ===== Valgrind: All Features =====
%%bash
valgrind --tool=memcheck --error-exitcode=1 ./sgfnd --generate --large --tiled --nsfw strict --nsfw-thresh 0.5 2>&1 | tail -20

In [ ]:
# ===== Training Mode (optional) =====
# Requires a dataset URL. Uncomment and provide your dataset.
%%bash
# ./sgfnd --train --url https://example.com/dataset.zip --epochs 5 --generate --nsfw enabled
echo 'Training mode available. Provide a dataset URL to use.'

## Project Structure

```
sgfnd/
├── include/sgfnd_core.h          # Public API
├── src/
│   ├── main.c                    # CLI entry, argument parsing, NSFW integration
│   ├── bridges/bridge_engine.c   # Bridge processing & compression
│   ├── color/color_grader.c      # HSV-based color grading
│   ├── generative/
│   │   ├── latent_diffusion.c    # MLP + DDPM diffusion
│   │   └── model_training.c      # Dataset, VAE, training loop
│   ├── io/image_io.c             # Tiled image I/O, rendering
│   ├── large_model/large_model.c # 30K bridge streaming
│   ├── safety/nsfw_filter.c      # NSFW prompt/image detection & sanitization
│   └── training/training_bot.c   # Auto-fetch, augment, train
├── tools/raw_to_image.py         # Raw→PNG converter
└── Makefile
```

## Key Features

- **Bridge Engine**: 92 initial bridges → 46 compressed via spectral analysis
- **Diffusion**: 512-dim latent, 1000 timesteps, MLP denoiser (SiLU activations)
- **Tiled Rendering**: 64×64 tiles, streaming to disk (low VRAM)
- **Large Model**: 30,000 bridges with on-demand VRAM streaming
- **Training Bot**: Auto-fetches images, augments, trains diffusion
- **Quantization**: INT8/INT4 bridge quantization support
- **NSFW Filter**: Keyword-based prompt blocking + pixel-analysis image flagging + block-blur sanitization

## Colab Notes

- Colab provides x86_64 CPU (and optional GPU) - builds natively
- Build takes ~30s, generation ~10-30s depending on mode
- Output raw files are 512×512×4×1byte = 1MB each
- For GPU acceleration, the CUDA diffusion code would need to be enabled
- NSFW filter runs entirely on CPU, minimal overhead
- Use Runtime → Change runtime type → GPU if you want GPU acceleration